# 类型声明生成与包分发

学习目标：生成与模块格式一致的类型声明，打包后由独立消费者验证真实安装入口。

前置知识：ESM 与 CommonJS、声明文件、模块解析、npm 包与项目配置。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；使用 ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/24-package-distribution/。

1. [package/src/index.ts](scripts/24-package-distribution/package/src/index.ts)：ES 模块实现与公共接口。
2. [package/src/modern.mts](scripts/24-package-distribution/package/src/modern.mts)：显式 ESM 入口。
3. [package/src/legacy.cts](scripts/24-package-distribution/package/src/legacy.cts)：独立的 CommonJS 实现。
4. [package/package.json](scripts/24-package-distribution/package/package.json)：发布文件与导入条件。
5. [tsconfig.json](scripts/24-package-distribution/tsconfig.json)：实现与声明输出；仅声明配置为 tsconfig.types.json。
6. [consumer/](scripts/24-package-distribution/consumer/)：独立消费者的配置、正常入口和类型反例。
7. [pack-check.mjs](scripts/24-package-distribution/pack-check.mjs)：本地打包、安装、检查和清理。

## 1 公共实现与声明生成

声明文件描述调用者可见的类型契约，不包含函数体。下面 greet 的参数和返回值构成公共接口；实现内部的默认值计算留在 JavaScript。declaration 同时生成 JavaScript 和声明，declarationMap 生成从声明跳转源码的映射。

本例主动标明返回值，避免公共接口无意跟随内部推断变化。types 为空，公共类型不依赖 Node 的全局声明。

以下片段来自 package/src/index.ts。

```typescript
export interface GreetingOptions { prefix?: string }
export function greet(name: string, options: GreetingOptions = {}): string {
  return (options.prefix ?? "你好") + "，" + name;
}
```

以下片段来自 tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [],
    "rootDir": "package/src",
    "outDir": "package/dist",
    "noEmitOnError": true,
    "declaration": true,
    "declarationMap": true
  },
  "files": [
    "package/src/index.ts",
    "package/src/modern.mts",
    "package/src/legacy.cts"
  ]
}
```

Step 1：检查包源码。

```bash
npm run check:24
# 无类型诊断。
```

Step 2：生成实现和声明。

```bash
npm run build:24
# 输出到 scripts/24-package-distribution/package/dist。
```

## 2 区分三种声明扩展名

NodeNext 下，.mts 生成 .mjs 与 .d.mts，.cts 生成 .cjs 与 .d.cts；普通 .ts 生成 .js 与 .d.ts，其模块格式还受 package.json 的 type 影响。本包 type 为 module，因此 index.d.ts 表示 ESM 声明。

现代入口重新导出同一个实现；CommonJS 入口提供对应的无状态函数。这里刻意不共享模块级可变状态；同时支持两套入口时，不能假定两次加载天然共享同一个状态对象。类型声明的模块格式应与它对应的 JavaScript 入口一致。

以下片段来自 package/src/modern.mts。

```typescript
export { greet } from "./index.js";
export type { GreetingOptions } from "./index.js";
```

以下片段来自 package/src/legacy.cts。

```typescript
export interface GreetingOptions { prefix?: string }
export function greet(name: string, options: GreetingOptions = {}): string {
  return (options.prefix ?? "你好") + "，" + name;
}
```

## 3 仅生成声明与源码映射

emitDeclarationOnly 不生成 JavaScript，适合由另一工具负责运行时代码转换时输出声明。它不等价于 noEmit，也不能单靠 .d.ts 让包在 Node.js 中运行。这里输出到独立 .types，和后面打包的 dist 分开。

declarationMap 只是源码定位信息，编辑器能否跳转还取决于源码位置是否可访问。本包把 src 一并纳入打包文件，因此映射中的 ../src 路径可以落到包内源码；消费者类型检查仍必须解析 dist 中的声明。

以下片段来自 tsconfig.types.json。

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": {
    "emitDeclarationOnly": true,
    "outDir": ".types"
  }
}
```

Step 1：单独生成声明。

```bash
npm run types:24
# scripts/24-package-distribution/.types 中有三种声明及其 .map，没有 JavaScript。
```

## 4 types、exports 和版本条件

每种消费者需要一组相互匹配的声明与实现，先按条件选分支，再确认两个文件都实际存在。

types 提供包的声明入口。exports 限制可访问的子路径并分配条件入口：本例 import 与 require 分支各有自己的 types，且 types 放在对应分支的 default 之前。Node.js 使用运行条件找到 JavaScript，TypeScript 在相应解析条件下读取声明。

./plain 是显式开放的子路径；不能因为 dist 文件已打包就任意从包名加内部路径导入。types 不负责运行时加载，exports 也不会替你创建尚未生成的文件。

typesVersions 根据 TypeScript 版本重定向声明，适合旧解析流程的兼容分发；当解析流程读取 exports 时，不再读取 typesVersions。需要版本化的 exports 类型分支时可使用 types@版本范围条件，不能把两种机制当作同时叠加的兜底。本例只有固定版本消费者，未声称兼容旧编译器。

![条件导出把声明与运行文件配成对。以 package.json 的根入口为例；消费者按包名访问。](image/illustration/24-01-package-condition-pairs.svg)

图示说明（依据篇末官方文档自绘）：图只展开本例根入口的 import/require 分支；不把 types 字段当作运行时实现。

对照下面 exports 的嵌套层级，再在独立消费者实验中核对实际解析到的声明与 JavaScript 路径。

以下片段来自 package/package.json。

```json
{
  "name": "notebook-greeting-ts-c",
  "version": "1.0.0",
  "private": true,
  "type": "module",
  "types": "./dist/index.d.ts",
  "files": [
    "dist",
    "src"
  ],
  "exports": {
    ".": {
      "import": {
        "types": "./dist/modern.d.mts",
        "default": "./dist/modern.mjs"
      },
      "require": {
        "types": "./dist/legacy.d.cts",
        "default": "./dist/legacy.cjs"
      }
    },
    "./plain": {
      "types": "./dist/index.d.ts",
      "default": "./dist/index.js"
    }
  }
}
```

## 5 发布文件和类型依赖

files 决定额外打包内容，package.json 等 npm 固定保留文件另有规则。这里发布 dist 与用于映射的 src，不发布消费者测试或工具缓存。private 为 true 阻止误发布，但仍可 npm pack 生成本地 tarball。

若公共 .d.ts 引用了第三方类型包，消费者也必须能解析它；应把实际公开依赖放在发布包的 dependencies，而不是只放开发依赖。编译器和测试工具属于开发环境；本包公共 API 只使用标准类型，因此没有外部类型依赖。不要为消除声明错误随意添加宽泛 declare module。

独立消费者的 types 为空，不依赖课程中自动载入的 @types。DOM lib 只用于此小型控制台例子的 console 声明，不提供或调用浏览器 DOM。

以下片段来自 consumer/tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025",
      "DOM"
    ],
    "types": [],
    "rootDir": ".",
    "outDir": "build",
    "noEmitOnError": true
  },
  "files": [
    "main.ts",
    "main.cts"
  ]
}
```

## 6 从安装包消费 ESM 和 CommonJS

消费者以包名导入，不以相对路径接触课程源码。ESM 入口同时检查默认入口和 ./plain；CommonJS 消费者使用 import = require，编译成 .cjs。显式变量类型验证声明契约，直接打印两种入口的调用结果；完整实验驱动再核对输出。

以下片段来自 consumer/main.ts。

```typescript
import { greet, type GreetingOptions } from "notebook-greeting-ts-c";
import { greet as plain } from "notebook-greeting-ts-c/plain";
const options: GreetingOptions = { prefix: "欢迎" };
const result: string = greet("读者", options);
// 按包名调用主入口和子路径；预期输出分别为“欢迎，读者”和“你好，读者”。
console.log("ESM installed OK", result); // → ESM installed OK 欢迎，读者
console.log("plain installed OK", plain("读者")); // → plain installed OK 你好，读者
```

以下片段来自 consumer/main.cts。

```typescript
import greeting = require("notebook-greeting-ts-c");
const result: string = greeting.greet("读者");
console.log("CJS installed OK", result); // → CJS installed OK 你好，读者
```

以下片段来自 consumer/type-errors.ts。

```typescript
import { greet } from "notebook-greeting-ts-c";
greet(7);
```

## 7 打包、临时安装与解析位置核对

pack-check.mjs 是本章可重复的本地发布实验。先构建、npm pack，再把 consumer 样例复制到单独临时目录，使用 tarball 路径安装；不使用源码目录链接。--offline、--ignore-scripts 和无依赖包让过程不依赖外网或安装脚本。

下面的驱动按构建、安装、检查声明和运行消费者四步展开。run 使用 execFileSync，命令失败直接抛出；--listFiles 核对安装声明，两个 resolve 调用显示运行入口。finally 限定清理范围后删除本次临时目录。

以下片段来自 pack-check.mjs。

```javascript
import assert from "node:assert/strict";
import { execFileSync } from "node:child_process";
import { mkdtempSync, realpathSync } from "node:fs";
import { cp, rm } from "node:fs/promises";
import { dirname, resolve, sep } from "node:path";

const chapter = import.meta.dirname;
const course = resolve(chapter, "../..");
const temp = mkdtempSync(resolve(chapter, "ts-c-24-pack-"));
const consumer = resolve(temp, "consumer");
const compiler = resolve(course, "node_modules/typescript/bin/tsc");
const npm = resolve(dirname(process.execPath), "node_modules/npm/bin/npm-cli.js");

// 这些步骤都应成功；启动失败或非零退出直接抛出，不重复包装错误。
function run(script, args, cwd) {
  return execFileSync(process.execPath, [script, ...args], {
    cwd, encoding: "utf8", windowsHide: true,
  });
}

try {
  // 1. 构建实现与声明，再把真实 tarball 放入本次临时目录。
  run(compiler, ["-p", chapter], course);
  const [packed] = JSON.parse(run(npm, ["pack", "./package", "--json", "--ignore-scripts", "--offline",
    "--cache", resolve(temp, "cache"), "--pack-destination", temp], chapter));

  // 2. 复制独立消费者，从 tarball 安装；缓存和安装产物都留在 temp 内。
  await cp(resolve(chapter, "consumer"), consumer, { recursive: true });
  run(npm, ["install", resolve(temp, packed.filename), "--offline", "--ignore-scripts", "--no-audit",
    "--no-fund", "--package-lock=false", "--cache", resolve(temp, "cache")], consumer);

  // 3. 消费者按包名编译，检查实际载入的是安装包的声明文件。
  const listed = run(compiler, ["-p", consumer, "--listFiles"], consumer).replaceAll("\\", "/");
  const declarations = ["modern.d.mts", "legacy.d.cts", "index.d.ts"];
  for (const file of declarations) assert.ok(listed.includes("node_modules/notebook-greeting-ts-c/dist/" + file));
  console.log("installed declarations:", declarations.join(", ")); // → installed declarations: modern.d.mts, legacy.d.cts, index.d.ts

  // 4. 打印实际运行入口，再执行生成的消费者；只保留关键结果核对。
  // → installed runtime: 后接 file: URL，末尾为 notebook-greeting-ts-c/dist/modern.mjs。
  // 临时目录名每次变化，不比较绝对路径。
  console.log("installed runtime:", run("--input-type=module",
    ["-e", "console.log(import.meta.resolve('notebook-greeting-ts-c'))"], consumer).trim());
  // → installed CJS runtime: 后接本机路径，末尾为 notebook-greeting-ts-c/dist/legacy.cjs。
  // Windows 路径分隔符为反斜线。
  console.log("installed CJS runtime:", run("--input-type=commonjs",
    ["-e", "console.log(require.resolve('notebook-greeting-ts-c'))"], consumer).trim());
  const esm = run(resolve(consumer, "build/main.js"), [], consumer).trim();
  const cjs = run(resolve(consumer, "build/main.cjs"), [], consumer).trim();
  assert.equal(esm, "ESM installed OK 欢迎，读者\nplain installed OK 你好，读者");
  assert.equal(cjs, "CJS installed OK 你好，读者");
  console.log(esm); // 两行：ESM installed OK 欢迎，读者；plain installed OK 你好，读者
  console.log(cjs); // → CJS installed OK 你好，读者
  console.log("pack 24 OK"); // → pack 24 OK；以上步骤均成功后才出现。
} finally {
  // 路径和前缀核对只用于限定递归清理范围，原始失败仍向调用方传播。
  const checked = realpathSync(temp);
  assert.ok(checked.startsWith(realpathSync(chapter) + sep));
  assert.ok(checked.split(sep).at(-1).startsWith("ts-c-24-pack-"));
  await rm(checked, { recursive: true, force: true });
}
```

Step 1：执行真实打包消费者实验。

```bash
npm run run:24
# 输出 installed declarations、ESM installed OK、CJS installed OK 和 pack 24 OK；临时安装目录随后删除。
```

Step 2：单独检查源码类型反例。

```bash
npm run errors:24
# 退出 1，TS2345：数值不能传给字符串参数。
```

单独检查消费者反例（选学）：正常打包实验不会运行 consumer/type-errors.ts，且结束时会删除临时安装。要观察它对已安装声明的诊断，先完成前文 npm run build:24，再在本章工作目录的 PowerShell 中按下面步骤准备一份独立消费者。开始时 .consumer-errors 目录不存在；安装、缓存与后续清理只涉及这份副本。

Step 1：复制本章消费者样例。

```powershell
Copy-Item scripts/24-package-distribution/consumer scripts/24-package-distribution/.consumer-errors -Recurse
```

Step 2：将已构建的本地包安装到副本中。

```powershell
npm install --prefix scripts/24-package-distribution/.consumer-errors ./scripts/24-package-distribution/package --install-links --offline --ignore-scripts --no-audit --no-fund --package-lock=false --cache scripts/24-package-distribution/.consumer-errors/.npm-cache
# --install-links 将本地目录打包后安装为副本；本例安装 1 个包，不创建源码链接。
```

Step 3：只检查副本中的类型反例。

```powershell
node node_modules/typescript/bin/tsc -p scripts/24-package-distribution/.consumer-errors/tsconfig.errors.json
# 预期退出码 1，只报告 TS2345：数值不能传给字符串参数；noEmit 不生成 JavaScript。
```

Step 4：检查结束后删除本次消费者副本及其安装、缓存。

```powershell
Remove-Item -LiteralPath scripts/24-package-distribution/.consumer-errors -Recurse
```

## 本章小结

- 声明与实现分别服务于检查器和运行时，模块格式与入口路径必须对应。
- exports 决定可访问入口，typesVersions 有自己的解析适用条件。
- 包是否可用要由实际安装后的消费者检查；源码能运行不能证明 tarball 完整。

## 练习

1. 给 GreetingOptions 增加 suffix，默认空字符串；ESM 和 CJS 实现都支持它，两类安装消费者均核对带后缀输出，并同步 pack-check.mjs 中相应的输出断言。
2. 临时从 files 排除 dist 后打包，观察安装消费者的编译失败；恢复后重新运行确认成功。
3. 在 consumer/main.ts 中把 greet 的名字实参改成数值，再运行 run:24，确认临时安装消费者的构建因 TS2345 失败，而非只能在源码检查时发现。

### 提示

1. 修改两份实现的接口及返回表达式，重新生成声明后再打包。
2. 调整 files 只保留 src；exports 中仍指向 dist。
3. 修改实际进入消费者正常配置的 main.ts；独立 type-errors.ts 不在 run:24 的正常项目里。

### 参考解析

1. 增加 `suffix?: string`，在结果末尾连接 `options.suffix ?? ""`；ESM/CJS 调用传同一后缀，打包驱动的预期字符串也随之变化。只改源码但不重新生成，安装包仍可能得到旧声明。
2. dist 缺失后，exports 仍指向不存在的声明与实现，安装消费者无法完成正常编译；恢复 files 后重新运行。
3. 临时消费者从已安装包的声明读到 name: string，因此数字实参被拒绝。检查完恢复 main.ts，正常打包流程才能重新到达 pack 24 OK。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [declaration](https://www.typescriptlang.org/tsconfig/declaration.html)、[declarationMap](https://www.typescriptlang.org/tsconfig/declarationMap.html)、[emitDeclarationOnly](https://www.typescriptlang.org/tsconfig/emitDeclarationOnly.html)；[Modules Reference](https://www.typescriptlang.org/docs/handbook/modules/reference.html#node16-nodenext) 的格式检测、exports、typesVersions；[Publishing](https://www.typescriptlang.org/docs/handbook/declaration-files/publishing.html#dependencies)：输出与公共类型依赖。 |
| Node.js 24.11.0 | [fsPromises.rm](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisesrmpath-options)：临时目录清理；[fsPromises.cp](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisescpsrc-dest-options)：递归复制及等待完成；[包入口及条件导出](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html#conditional-exports)；[import.meta.resolve](https://nodejs.org/download/release/v24.11.0/docs/api/esm.html#importmetaresolvespecifier)、[require.resolve](https://nodejs.org/download/release/v24.11.0/docs/api/modules.html#requireresolverequest-options)、[mkdtempSync](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fsmkdtempsyncprefix-options)、[execFileSync](https://nodejs.org/download/release/v24.11.0/docs/api/child_process.html#child_processexecfilesyncfile-args-options)：入口解析、临时目录与子进程检查。 |
| npm 11 | [npm pack](https://docs.npmjs.com/cli/v11/commands/npm-pack/)、[本地安装](https://docs.npmjs.com/cli/v11/commands/npm-install/#install-links)、[package.json / files](https://docs.npmjs.com/cli/v11/configuring-npm/package-json/#files)：本地 tarball、安装参数与打包文件。 |
| Microsoft Learn | [Copy-Item：复制目录](https://learn.microsoft.com/en-us/powershell/module/microsoft.powershell.management/copy-item?view=powershell-7.5#example-3-copy-directory-and-contents-to-a-new-directory)、[Remove-Item / Recurse](https://learn.microsoft.com/en-us/powershell/module/microsoft.powershell.management/remove-item?view=powershell-7.5#-recurse)：PowerShell 中建立与清理本次消费者副本。 |